# Combining Datasets with Bead-Based Alignment

pyMINFLUX's combiner merges split-exchange MINFLUX acquisitions by computing a rigid transformation between each pair of datasets using shared fiducial beads tracked in the MBM (bead monitor) channel.

## Imports

In [1]:
from pathlib import Path
from pyminflux.reader import MinFluxReaderV2
from pyminflux.processor import MinFluxDataset, MinFluxProcessor
from pyminflux.combiner import combine_datasets_with_bead_alignment

## Load datasets

Each sub-folder is a Zarr store from one acquisition cycle. `MinFluxDataset.from_reader()` wraps the reader into the container expected by the combiner.

In [2]:
base_dir = Path("Potential Included Datasets/NupsQC_2D_FakeSplitExchange")

cycle_paths = [
    base_dir / "260320-132517_minflux2D_Nups_Cycle1",
    base_dir / "260320-133051_minflux2D_Nups_Cycle2",
    base_dir / "260320-133650_minflux2D_Nups_Cycle3",
]

datasets = [MinFluxDataset.from_reader(MinFluxReaderV2(path))
            for path in cycle_paths]

Read 6 beads (4 used).
Read 6 beads (4 used).
Read 6 beads (4 used).


In [3]:
for i, ds in enumerate(datasets, 1):
    beads = list(ds.mbm_data.get("mbm", {}).keys()) if ds.mbm_data else []
    print(f"Cycle {i} — MBM beads: {beads}")

Cycle 1 — MBM beads: ['R11', 'R2', 'R20', 'R25', 'R9', 'R10']
Cycle 2 — MBM beads: ['R11', 'R2', 'R20', 'R25', 'R9', 'R10']
Cycle 3 — MBM beads: ['R11', 'R2', 'R20', 'R25', 'R9', 'R10']


## Combine

Combining more than two datasets is done iteratively: cycle 2 is aligned onto cycle 1, then cycle 3 is aligned onto the result. The combiner estimates a rigid transformation from shared bead positions and assigns each cycle a unique fluorophore ID (`fluo`).

In [4]:
# combine two datasets with bead alignment
combined_12 = combine_datasets_with_bead_alignment(
    datasets[0], datasets[1]
)

# combine all three datasets sequentially
combined_all = datasets[0]
for ds in datasets[1:]:
    combined_all = combine_datasets_with_bead_alignment(combined_all, ds)

Using 4 bead pairs for alignment
Aligning using 4 bead positions
Alignment completed using rigid (rotation + translation) mode.
Mean residual: 1.08 nm.
Mean residual before alignment: 4.05 nm.
Using 4 bead pairs for alignment
Aligning using 4 bead positions
Alignment completed using rigid (rotation + translation) mode.
Mean residual: 1.08 nm.
Mean residual before alignment: 4.05 nm.
Using 4 bead pairs for alignment
Aligning using 4 bead positions
Alignment completed using rigid (rotation + translation) mode.
Mean residual: 1.18 nm.
Mean residual before alignment: 2.92 nm.


### Manual bead correspondences

By default the combiner matches beads by name. If bead names differ across cycles — or you want to restrict alignment to a specific subset — pass a `bead_correspondence` dict that maps each **moving** bead name to the corresponding **reference** bead name.

In [5]:
# Map moving-bead name → reference-bead name
bead_correspondence = {
    "R11": "R11",
    "R25": "R25",
    "R9": "R9",
}

combined_12_manual = combine_datasets_with_bead_alignment(
    datasets[0], datasets[1],
    bead_correspondence=bead_correspondence,
)

Using 3 bead pairs for alignment
Aligning using 3 bead positions
Alignment completed using rigid (rotation + translation) mode.
Mean residual: 1.12 nm.
Mean residual before alignment: 4.15 nm.


## Further processing

A `MinFluxProcessor` object can be constructed from the `MinFluxDataset` resulting from the combination. Have a look at the "processing.ipynb" example for more processing examples.

In [6]:
processor = MinFluxProcessor(combined_all, min_trace_length=4)

df_filtered = processor.filtered_dataframe
df_filtered.head()

/Users/albertm/.pixi/envs/conda/envs/pyminflux/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,tid,tim,x,y,z,efo,cfr,eco,dcr,dwell,fluo,fbg,iid
0,428,16.693998,1511.232477,-3312.865467,-0.000059,66489.148438,0.814941,206,0.623047,3.0,1,0.0,1
1,428,16.706161,1509.850062,-3317.464291,-0.000059,61002.179688,0.791504,189,0.616699,3.0,1,0.0,2
2,428,16.718322,1516.353667,-3319.932626,-0.000060,89728.070312,0.959961,278,0.594238,3.0,1,0.0,3
3,428,16.728060,1517.660723,-3317.614333,-0.000060,177196.796875,0.939453,366,0.560547,2.0,1,0.0,4
4,428,16.737809,1514.829760,-3315.502395,-0.000060,182038.250000,0.785645,188,0.648926,1.0,1,0.0,5
